# Seguimiento mensual — pronóstico y lista de gestión contra la realidad

Este notebook responde dos preguntas cada mes, una vez que llega el archivo nuevo:

1. **¿Qué tan bien pronosticó el modelo los meses que ya pasaron?** Toma cada pronóstico
   guardado en `historial_pronosticos/` (uno por fecha de corte) y lo compara con el consumo
   real de los meses que ya quedaron consolidados. Es la precisión *en vivo*, distinta del
   backtest: aquí el modelo no sabía nada del futuro.
2. **¿Qué pasó con los clientes que pusimos en la lista de gestión?** Toma cada lista guardada
   en `gestion_caida/historial/` y mira si esos clientes siguieron cayendo, se estabilizaron o
   se recuperaron en los meses siguientes, por trayectoria y severidad.

Se corre **después** de `Priorizacion_gestion_caida.ipynb`. Recalcula todo desde cero en cada
corrida (no acumula), así que se puede repetir sin problema.

Solo evalúa meses **consolidados** (según `utilidades_borde`): un mes provisional no es una
verdad contra la que se pueda medir nada.

In [ ]:
# ============================================================
# 1. LIBRERÍAS Y RUTAS
# ============================================================

from pathlib import Path
import os
import re
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from utilidades_borde import ultimo_periodo_consolidado

BASE_DIR = Path(os.environ.get("EBSA_DATOS", r"C:\Users\Home\Documents\Datos_Ebsa"))

PREPROC_DIR = BASE_DIR / "03_serie_modelado"
SALIDA_MODELO_DIR = BASE_DIR / "04_pronostico" / "modelo_final"
HISTORIAL_PRONOSTICOS_DIR = SALIDA_MODELO_DIR / "historial_pronosticos"
GESTION_DIR = BASE_DIR / "07_gestion_caida"
HISTORIAL_GESTION_DIR = GESTION_DIR / "historial"

SEGUIMIENTO_DIR = BASE_DIR / "08_seguimiento"
SEGUIMIENTO_DIR.mkdir(parents=True, exist_ok=True)

RUTA_SERIE = PREPROC_DIR / "serie_mensual_modelado_preprocesada.parquet"
RUTA_METRICAS_BACKTEST = SALIDA_MODELO_DIR / "metricas_sistema_por_perfil_horizonte_optimizado.csv"

RUTA_SEG_PERFIL = SEGUIMIENTO_DIR / "seguimiento_pronostico_por_perfil.csv"
RUTA_SEG_GLOBAL = SEGUIMIENTO_DIR / "seguimiento_pronostico_global.csv"
RUTA_SEG_ZONA = SEGUIMIENTO_DIR / "seguimiento_pronostico_por_zona.csv"
RUTA_SEG_GESTION = SEGUIMIENTO_DIR / "seguimiento_lista_gestion.csv"
RUTA_SEG_GESTION_DETALLE = SEGUIMIENTO_DIR / "seguimiento_lista_gestion_detalle.parquet"

HORIZONTES = [1, 2, 3, 4, 5, 6]

# Un cliente de la lista se considera RECUPERADO si su consumo real posterior
# vuelve al menos a este porcentaje de lo que consumía ANTES de caer, y
# SIGUE_CAYENDO si queda por debajo de este porcentaje de su consumo reciente.
PCT_RECUPERACION = 0.90
PCT_SIGUE_CAYENDO = 0.90

print("Serie                :", RUTA_SERIE)
print("Historial pronósticos:", HISTORIAL_PRONOSTICOS_DIR)
print("Historial gestión    :", HISTORIAL_GESTION_DIR)
print("Salidas              :", SEGUIMIENTO_DIR)


In [ ]:
# ============================================================
# 2. SERIE REAL Y ÚLTIMO MES CONSOLIDADO
# ============================================================

if not RUTA_SERIE.exists():
    raise FileNotFoundError(f"No existe:\n{RUTA_SERIE}")

serie = pd.read_parquet(
    RUTA_SERIE, columns=["NIU", "periodo", "consumo_kwh_mensual", "es_rural"], engine="pyarrow",
)
serie["NIU"] = serie["NIU"].astype("string").str.strip()
serie["periodo"] = pd.to_datetime(serie["periodo"], errors="coerce")
serie["consumo_kwh_mensual"] = pd.to_numeric(serie["consumo_kwh_mensual"], errors="coerce").astype("float32")

periodo_max = serie["periodo"].max().to_period("M").to_timestamp()

PERIODO_CONSOLIDADO, N_PROVISIONALES, _ = ultimo_periodo_consolidado(serie, col_grupo="es_rural")

zona_por_niu = (
    serie.drop_duplicates("NIU")[["NIU", "es_rural"]]
    .assign(zona=lambda d: np.where(d["es_rural"].fillna(False).astype(bool), "RURAL", "URBANO"))
    .set_index("NIU")["zona"]
)

print(f"\nSe evalúa contra meses hasta {PERIODO_CONSOLIDADO:%Y-%m} (último consolidado).")


In [ ]:
# ============================================================
# 3. PRONÓSTICOS GUARDADOS vs. CONSUMO REAL
# ============================================================

patron_pred = re.compile(r"^predicciones_6_meses_corte_(\d{4}-\d{2})\.parquet$")

archivos_pred = sorted(
    (pd.Timestamp(m.group(1) + "-01"), ruta)
    for ruta in HISTORIAL_PRONOSTICOS_DIR.glob("predicciones_6_meses_corte_*.parquet")
    if (m := patron_pred.match(ruta.name))
)

print("PRONÓSTICOS GUARDADOS")
print("-" * 70)
if not archivos_pred:
    print("No hay pronósticos en", HISTORIAL_PRONOSTICOS_DIR)
    print("Se crean al correr Backtest_y_reentrenamiento_final_optimizado.ipynb.")
for corte, ruta in archivos_pred:
    evaluables = [h for h in HORIZONTES if corte + pd.DateOffset(months=h) <= PERIODO_CONSOLIDADO]
    print(f"  corte {corte:%Y-%m}: horizontes evaluables hoy -> {evaluables or 'ninguno todavía'}")

# Meses reales que hacen falta
meses_necesarios = sorted({
    corte + pd.DateOffset(months=h)
    for corte, _ in archivos_pred for h in HORIZONTES
    if corte + pd.DateOffset(months=h) <= PERIODO_CONSOLIDADO
})

real = (
    serie[serie["periodo"].isin(meses_necesarios)][["NIU", "periodo", "consumo_kwh_mensual"]]
    .rename(columns={"periodo": "fecha_target", "consumo_kwh_mensual": "real_kwh"})
    if meses_necesarios else pd.DataFrame(columns=["NIU", "fecha_target", "real_kwh"])
)

partes = []
for corte, ruta in archivos_pred:
    pred = pd.read_parquet(ruta, engine="pyarrow")
    pred["NIU"] = pred["NIU"].astype("string").str.strip()
    for h in HORIZONTES:
        fecha_target = corte + pd.DateOffset(months=h)
        if fecha_target > PERIODO_CONSOLIDADO:
            continue
        col_pred = f"pred_{h}m_kwh"
        if col_pred not in pred.columns:
            continue
        temp = pred[["NIU", "perfil", col_pred]].rename(columns={col_pred: "pred_kwh"})
        temp["fecha_corte"] = corte
        temp["horizonte"] = h
        temp["fecha_target"] = fecha_target
        temp["fecha_corte_modelo"] = (
            pd.to_datetime(pred["fecha_corte_modelo"]).iloc[0]
            if "fecha_corte_modelo" in pred.columns else pd.NaT
        )
        partes.append(temp)

if partes:
    evaluacion = pd.concat(partes, ignore_index=True).merge(real, on=["NIU", "fecha_target"], how="left")
    evaluacion["zona"] = evaluacion["NIU"].map(zona_por_niu).fillna("URBANO")
    sin_real = evaluacion["real_kwh"].isna()
    print(f"\nPares pronóstico-real: {len(evaluacion):,} | sin dato real (cliente sin fila ese mes): "
          f"{int(sin_real.sum()):,} -> excluidos de las métricas")
    evaluacion = evaluacion[~sin_real].copy()
else:
    evaluacion = pd.DataFrame(columns=["NIU", "perfil", "pred_kwh", "fecha_corte", "horizonte",
                                       "fecha_target", "fecha_corte_modelo", "real_kwh", "zona"])
    print("\nTodavía no hay ningún mes pronosticado que ya esté consolidado: no hay nada que evaluar.")
    print("Volverá a intentarlo el mes que viene.")


In [ ]:
# ============================================================
# 4. MÉTRICAS EN VIVO: POR PERFIL, GLOBAL Y POR ZONA
# ============================================================

def metricas(grupo):
    real = grupo["real_kwh"].to_numpy(dtype="float64")
    pred = grupo["pred_kwh"].to_numpy(dtype="float64")
    total = real.sum()
    return pd.Series({
        "n": len(grupo),
        "real_total_kwh": total,
        "pred_total_kwh": pred.sum(),
        "WAPE_pct": (np.abs(real - pred).sum() / total * 100) if total > 0 else np.nan,
        "sesgo_pct": ((pred.sum() - total) / total * 100) if total > 0 else np.nan,
        "MAE_kwh": np.abs(real - pred).mean(),
    })


def resumir(df, claves):
    if df.empty:
        return pd.DataFrame(columns=claves + ["n", "real_total_kwh", "pred_total_kwh",
                                              "WAPE_pct", "sesgo_pct", "MAE_kwh"])
    out = df.groupby(claves).apply(metricas, include_groups=False).reset_index()
    out["n"] = out["n"].astype(int)
    return out


seg_perfil = resumir(evaluacion, ["fecha_corte", "horizonte", "perfil"])
seg_global = resumir(evaluacion, ["fecha_corte", "horizonte"])
seg_zona = resumir(evaluacion, ["fecha_corte", "horizonte", "zona"])

# Referencia: lo que dio el backtest para cada perfil x horizonte
if RUTA_METRICAS_BACKTEST.exists():
    bt = pd.read_csv(RUTA_METRICAS_BACKTEST)[["perfil", "horizonte", "WAPE_final_pct"]]
    bt = bt.rename(columns={"WAPE_final_pct": "WAPE_backtest_pct"})
    seg_perfil = seg_perfil.merge(bt, on=["perfil", "horizonte"], how="left")
    seg_perfil["dif_vs_backtest_pp"] = seg_perfil["WAPE_pct"] - seg_perfil["WAPE_backtest_pct"]

for df in (seg_perfil, seg_global, seg_zona):
    if not df.empty:
        df["fecha_corte"] = pd.to_datetime(df["fecha_corte"]).dt.strftime("%Y-%m")
        df["evaluado_con_datos_hasta"] = f"{PERIODO_CONSOLIDADO:%Y-%m}"

seg_perfil.round(3).to_csv(RUTA_SEG_PERFIL, index=False, encoding="utf-8-sig")
seg_global.round(3).to_csv(RUTA_SEG_GLOBAL, index=False, encoding="utf-8-sig")
seg_zona.round(3).to_csv(RUTA_SEG_ZONA, index=False, encoding="utf-8-sig")

print("PRECISIÓN EN VIVO — GLOBAL POR CORTE Y HORIZONTE")
print("=" * 78)
if seg_global.empty:
    print("(sin meses evaluables todavía)")
else:
    display(seg_global.round(2))
    print("\nPOR PERFIL (con el WAPE del backtest como referencia)")
    print("-" * 78)
    display(seg_perfil.round(2))
    print("\nPOR ZONA")
    print("-" * 78)
    display(seg_zona.round(2))
    print("\nCÓMO LEERLO")
    print("  WAPE_pct  : error porcentual ponderado del pronóstico contra lo que realmente pasó.")
    print("  sesgo_pct : positivo = el modelo pronosticó de más; negativo = de menos.")
    print("  dif_vs_backtest_pp : si es claramente positiva (peor que el backtest) durante")
    print("                       varios meses seguidos, toca reentrenar.")

print("\nGuardado:")
for r in (RUTA_SEG_GLOBAL, RUTA_SEG_PERFIL, RUTA_SEG_ZONA):
    print(" •", r)


In [ ]:
# ============================================================
# 5. GRÁFICA: WAPE EN VIVO POR HORIZONTE, UNA LÍNEA POR CORTE
# ============================================================

if not seg_global.empty:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for corte, g in seg_global.groupby("fecha_corte"):
        g = g.sort_values("horizonte")
        ax.plot(g["horizonte"], g["WAPE_pct"], marker="o", label=f"corte {corte}")
    if RUTA_METRICAS_BACKTEST.exists():
        bt_global = SALIDA_MODELO_DIR / "metricas_sistema_ganador_backtest_optimizado.csv"
        if bt_global.exists():
            b = pd.read_csv(bt_global).sort_values("horizonte")
            ax.plot(b["horizonte"], b["WAPE_final_pct"], linestyle="--", color="gray",
                    marker="x", label="backtest (referencia)")
    ax.set_xlabel("Horizonte (meses)")
    ax.set_ylabel("WAPE (%)")
    ax.set_title("Precisión en vivo del pronóstico contra el consumo real")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Sin datos para graficar todavía.")


In [ ]:
# ============================================================
# 6. ¿QUÉ PASÓ CON LOS CLIENTES DE LA LISTA DE GESTIÓN?
# ============================================================
# Para cada lista guardada (un corte) se mira el consumo real de sus clientes
# en los meses siguientes ya consolidados, y se clasifica:
#   RECUPERADO     : volvió al menos al 90% de lo que consumía ANTES de caer
#   SIGUE_CAYENDO  : quedó por debajo del 90% de su consumo reciente (siguió bajando)
#   ESTABLE_BAJO   : ni lo uno ni lo otro (se quedó en el nivel caído)
# Cruzado con trayectoria y severidad, dice si las etiquetas del modelo
# discriminan de verdad, sin necesidad de esperar la visita en campo.
# ============================================================

patron_lista = re.compile(r"^gestion_caida_operativa_corte_(\d{4}-\d{2})\.csv$")
archivos_lista = sorted(
    (pd.Timestamp(m.group(1) + "-01"), ruta)
    for ruta in HISTORIAL_GESTION_DIR.glob("gestion_caida_operativa_corte_*.csv")
    if (m := patron_lista.match(ruta.name))
)

print("LISTAS DE GESTIÓN GUARDADAS")
print("-" * 70)
if not archivos_lista:
    print("No hay listas en", HISTORIAL_GESTION_DIR)
    print("Se crean al correr Priorizacion_gestion_caida.ipynb.")

meses_lista = sorted({
    corte + pd.DateOffset(months=h)
    for corte, _ in archivos_lista for h in HORIZONTES
    if corte + pd.DateOffset(months=h) <= PERIODO_CONSOLIDADO
})
real_lista = (
    serie[serie["periodo"].isin(meses_lista)][["NIU", "periodo", "consumo_kwh_mensual"]]
    .rename(columns={"periodo": "fecha_target", "consumo_kwh_mensual": "real_kwh"})
    if meses_lista else pd.DataFrame(columns=["NIU", "fecha_target", "real_kwh"])
)

partes = []
for corte, ruta in archivos_lista:
    lista = pd.read_csv(ruta, dtype={"NIU": "string", "ciclo_etiqueta": "string"})
    lista["NIU"] = lista["NIU"].str.strip()
    evaluables = [h for h in HORIZONTES if corte + pd.DateOffset(months=h) <= PERIODO_CONSOLIDADO]
    print(f"  corte {corte:%Y-%m}: {len(lista):,} clientes | meses posteriores evaluables: "
          f"{evaluables or 'ninguno todavía'}")
    cols = [c for c in ["NIU", "cluster_id", "zona", "tramo_consumo", "severidad", "veredicto",
                        "trayectoria", "estado_en_lista", "consumo_anterior_kwh",
                        "consumo_reciente_kwh", "perdida_kwh_mes", "valor_riesgo_mes"]
            if c in lista.columns]
    for h in evaluables:
        temp = lista[cols].copy()
        temp["fecha_corte"] = corte
        temp["meses_despues"] = h
        temp["fecha_target"] = corte + pd.DateOffset(months=h)
        partes.append(temp)

if partes:
    seguimiento_lista = pd.concat(partes, ignore_index=True).merge(
        real_lista, on=["NIU", "fecha_target"], how="left",
    )
    con_dato = seguimiento_lista["real_kwh"].notna()
    seguimiento_lista = seguimiento_lista[con_dato].copy()

    seguimiento_lista["resultado"] = np.select(
        [
            seguimiento_lista["real_kwh"] >= PCT_RECUPERACION * seguimiento_lista["consumo_anterior_kwh"],
            seguimiento_lista["real_kwh"] < PCT_SIGUE_CAYENDO * seguimiento_lista["consumo_reciente_kwh"],
        ],
        ["RECUPERADO", "SIGUE_CAYENDO"],
        default="ESTABLE_BAJO",
    )
    seguimiento_lista.to_parquet(RUTA_SEG_GESTION_DETALLE, index=False, engine="pyarrow")

    def tabla(claves):
        t = (
            seguimiento_lista.groupby(claves + ["resultado"]).size()
            .unstack("resultado", fill_value=0)
        )
        for col in ["RECUPERADO", "ESTABLE_BAJO", "SIGUE_CAYENDO"]:
            if col not in t.columns:
                t[col] = 0
        t["n"] = t[["RECUPERADO", "ESTABLE_BAJO", "SIGUE_CAYENDO"]].sum(axis=1)
        for col in ["RECUPERADO", "ESTABLE_BAJO", "SIGUE_CAYENDO"]:
            t[f"pct_{col.lower()}"] = (t[col] / t["n"] * 100).round(1)
        return t.reset_index()

    filas_resumen = []
    for claves, nombre in [
        (["fecha_corte", "meses_despues"], "TOTAL"),
        (["fecha_corte", "meses_despues", "trayectoria"], "trayectoria"),
        (["fecha_corte", "meses_despues", "severidad"], "severidad"),
        (["fecha_corte", "meses_despues", "zona"], "zona"),
    ]:
        if all(c in seguimiento_lista.columns for c in claves):
            t = tabla(claves)
            t["corte_por"] = nombre
            t["valor_corte"] = t[claves[-1]].astype(str) if nombre != "TOTAL" else "TOTAL"
            filas_resumen.append(t[["corte_por", "valor_corte", "fecha_corte", "meses_despues", "n",
                                    "RECUPERADO", "ESTABLE_BAJO", "SIGUE_CAYENDO",
                                    "pct_recuperado", "pct_estable_bajo", "pct_sigue_cayendo"]])
    resumen_lista = pd.concat(filas_resumen, ignore_index=True)
    resumen_lista["fecha_corte"] = pd.to_datetime(resumen_lista["fecha_corte"]).dt.strftime("%Y-%m")
    resumen_lista.to_csv(RUTA_SEG_GESTION, index=False, encoding="utf-8-sig")

    print("\nQUÉ PASÓ CON LA LISTA — TOTAL")
    print("=" * 78)
    display(resumen_lista[resumen_lista["corte_por"] == "TOTAL"])
    print("\nPOR TRAYECTORIA (la prueba de que CAIDA_ACELERANDO merece ir primero)")
    print("-" * 78)
    display(resumen_lista[resumen_lista["corte_por"] == "trayectoria"])
    print("\nPOR SEVERIDAD")
    print("-" * 78)
    display(resumen_lista[resumen_lista["corte_por"] == "severidad"])
    print("\nGuardado:", RUTA_SEG_GESTION)
    print("Detalle :", RUTA_SEG_GESTION_DETALLE)
else:
    resumen_lista = pd.DataFrame()
    print("\nTodavía no hay meses posteriores consolidados para ninguna lista: no hay nada que evaluar.")


In [ ]:
# ============================================================
# 7. CIERRE
# ============================================================

print("SEGUIMIENTO MENSUAL — TERMINADO")
print("=" * 78)
print(f"Datos consolidados hasta : {PERIODO_CONSOLIDADO:%Y-%m}")
print(f"Pronósticos guardados    : {len(archivos_pred)}  | pares evaluados: {len(evaluacion):,}")
print(f"Listas de gestión        : {len(archivos_lista)}  | clientes-mes evaluados: "
      f"{0 if resumen_lista.empty else int(resumen_lista.loc[resumen_lista['corte_por'] == 'TOTAL', 'n'].sum()):,}")
print("\nSalidas en", SEGUIMIENTO_DIR)
for r in (RUTA_SEG_GLOBAL, RUTA_SEG_PERFIL, RUTA_SEG_ZONA, RUTA_SEG_GESTION):
    print(" •", r.name, "(existe)" if r.exists() else "(sin datos todavía)")
